# Chapter 5 — Build Programs From Programs

**Book alignment:** DSPy From First Principles, Chapter 5

**Question this notebook isolates:** Which edges in the Analyze → Rewrite → Assess graph change the scored rewrite, and which stages merely observe it?


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

from common.data import teaching_cases
from common.dspy_program import EditorialRewriteProgram
from common.metrics import score_editorial_output


## Data flow is auditable plumbing

The composed program threads intermediate state between stages. Lineage checks confirm the plumbing: analysis outputs reach the rewrite unchanged, and the rewrite reaches the assessor unchanged. That proves data arrived — not that it mattered.


In [ ]:
ed002 = {c.case_id: c for c in teaching_cases()}["ed-002"]

# Recorded intermediate state for ed-002 (no LM is called).
REAL_ANALYSIS = {
    "issue_summary": "Redundant causal clause; compress while keeping the cause.",
    "preservation_notes": "Keep Mira, walking, rain and mud; no vehicle travel.",
}
rewrite_inputs = {
    "sentence": ed002.sentence,
    "goal": ed002.goal,
    "context": ed002.context,
    "issue_summary": REAL_ANALYSIS["issue_summary"],
    "preservation_notes": REAL_ANALYSIS["preservation_notes"],
}
REAL_REWRITE = "Mira was tired from walking all day in rain and mud."
assess_inputs = {
    "sentence": ed002.sentence,
    "rewritten_text": REAL_REWRITE,
    "goal": ed002.goal,
    "context": ed002.context,
}

lineage = {
    "issue_summary_reaches_rewrite": rewrite_inputs["issue_summary"] == REAL_ANALYSIS["issue_summary"],
    "preservation_notes_reach_rewrite": rewrite_inputs["preservation_notes"] == REAL_ANALYSIS["preservation_notes"],
    "rewrite_reaches_assess": assess_inputs["rewritten_text"] == REAL_REWRITE,
}
print(lineage)


In [ ]:
assert all(lineage.values())
print("plumbing passes; causality still unproven")


## REAL versus BLANK versus SHUFFLED

Hold sentence, goal, context, model, and signature fixed; vary only the two analysis fields entering the rewrite. The deterministic v1 metric scores fixed candidate rewrites, so the comparison isolates the causal contribution of intermediate state.


In [ ]:
CANDIDATES = {
    "REAL": "Mira was tired from walking all day in rain and mud.",
    "BLANK": ed002.sentence,
    "SHUFFLED": "Mira was tired from driving all day in rain and mud.",
}
scores = {name: score_editorial_output(ed002, text) for name, text in CANDIDATES.items()}
for name, breakdown in scores.items():
    print(f"{name:8s} score={breakdown.score:.3f} gate={breakdown.hard_gate_passed} failures={list(breakdown.failure_categories)}")


In [ ]:
assert scores["REAL"].score > scores["BLANK"].score > scores["SHUFFLED"].score
assert scores["SHUFFLED"].score == 0.0
assert "forbidden_term" in scores["SHUFFLED"].failure_categories
assert "unchanged" in scores["BLANK"].failure_categories
assert scores["BLANK"].score < scores["REAL"].score
print("correct intermediate analysis is causally useful on this fixture")


## The stage that pays for nothing

The assessor runs after `rewritten_text` is already fixed and emits `risk` plus `assessment` — neither of which any code path consumes before the result is returned. The program below is constructed, never executed; the dataflow argument needs no model call.


In [ ]:
program = EditorialRewriteProgram()
stages = sorted(name for name, _ in program.named_predictors())

# Structural fact about forward(): the returned sentence is rewrite output
# regardless of what the assessor claims. Same rewrite, two assessments.
returned_texts = []
for risk in ("low", "high"):
    assessment = {"risk": risk, "assessment": f"fixture assessment ({risk})"}
    returned = {"rewritten_text": REAL_REWRITE, "risk": assessment["risk"]}
    returned_texts.append(returned["rewritten_text"])
print("stages:", stages)
print("returned under risk=low :", returned_texts[0])
print("returned under risk=high:", returned_texts[1])


In [ ]:
assert set(stages) == {"analyze", "rewrite", "assess"}
assert returned_texts[0] == returned_texts[1] == REAL_REWRITE
print("no causal path: assess adds a call, observability, and cost - not control")


## What we earned

Decomposition is not automatically good: real intermediate analysis moves the rewrite, while a stage with no downstream consumer cannot change the scored output no matter how explicit or well named it is. Ask of every stage what consumes it, and price each call.

Notebook 06 / Chapter 6 stops treating the model as background: every number so far is a number about one configured dependency, so is the model part of the program, or a dependency of it?
